[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/05_Advanced_Topics/01_llava_architecture/01_llava_architecture.ipynb)

# 01. LLaVA Architecture: Connecting Vision to LLMs

**LLaVA (Large Language and Vision Assistant)** is the blueprint for modern multimodal LLMs.

**This notebook covers:**
- LLaVA architecture — step-by-step diagram
- The projection layer: how images become "language"
- Two-stage training: pretrain projector → finetune end-to-end
- Build a mini-LLaVA from scratch
- How GPT-4V, Gemini, etc. extend this idea

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/05_Advanced_Topics/01_llava_architecture")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

In [ ]:
# LLaVA Architecture Diagram

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('LLaVA Architecture', fontsize=20, fontweight='bold', pad=20)

# Image path
draw_architecture_block(ax, 3, 9, 3.5, 0.7, 'Input Image', '#E74C3C')
draw_architecture_block(ax, 3, 7.5, 3.5, 0.9, 'Vision Encoder\n(CLIP ViT-L, frozen)', '#E74C3C')
draw_architecture_block(ax, 3, 5.8, 3.5, 0.7, 'Image Features\n[N, 1024]', '#C0392B')
draw_architecture_block(ax, 3, 4.3, 3.5, 0.9, 'Projection (MLP)\nLinear→GELU→Linear\n[N, 1024] → [N, 4096]', '#F39C12')
ax.text(3, 3.3, 'Image "tokens"\n(same dim as text!)', ha='center', fontsize=10,
        color='#F39C12', fontweight='bold')

for y1, y2 in [(8.6, 8.0), (7.0, 6.2), (5.4, 4.8)]:
    draw_arrow(ax, (3, y1), (3, y2))

# Text path
draw_architecture_block(ax, 10, 9, 4, 0.7, 'User: "Describe this image"', '#3498DB')
draw_architecture_block(ax, 10, 7.5, 4, 0.7, 'Tokenize + Embed', '#3498DB')

draw_arrow(ax, (10, 8.6), (10, 8.0))

# Concatenation
draw_architecture_block(ax, 8, 5.5, 10, 0.9, 'Concatenate: [image_tokens, text_tokens]\n= [img1, img2, ..., imgN, user, describe, this, image]', '#9B59B6', fontsize=9)

draw_arrow(ax, (3, 3.8), (4.5, 6.0))
draw_arrow(ax, (10, 7.0), (10, 6.0))

# LLM
draw_architecture_block(ax, 8, 3.5, 10, 1.2, 'LLM Decoder (LLaMA, frozen or LoRA)\nCausal self-attention over all tokens', '#2ECC71', fontsize=11)
draw_architecture_block(ax, 8, 1.5, 6, 0.8, 'Generated Answer:\n"A cat sitting on a red sofa"', '#34495E')

draw_arrow(ax, (8, 4.9), (8, 4.2))
draw_arrow(ax, (8, 2.8), (8, 2.0))

# Annotations
ax.text(14.5, 7.5, 'Frozen', fontsize=10, color='#E74C3C', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#FADBD8', alpha=0.8))
ax.text(14.5, 4.3, 'Trainable\n(Stage 1)', fontsize=10, color='#F39C12', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#FEF9E7', alpha=0.8))
ax.text(14.5, 3.5, 'LoRA\n(Stage 2)', fontsize=10, color='#2ECC71', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#D5F5E3', alpha=0.8))

plt.tight_layout()
plt.savefig('../assets/llava_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

## The Key Insight: Images as Language Tokens

LLaVA's trick is simple but brilliant:
1. Use a frozen CLIP vision encoder to get image features
2. Use an MLP to project image features into the **same dimension as text tokens**
3. Concatenate image tokens + text tokens
4. Feed everything into the LLM — it treats image tokens just like word tokens!

## Tensor Flow Through LLaVA

Let's trace the exact tensor shapes through the LLaVA pipeline for a standard 224×224 input image and a text prompt of length $T$.

### Step 1: Image → Vision Encoder

Input image $I \in \mathbb{R}^{H \times W \times 3}$ with $H = W = 224$.

CLIP ViT-L/14 splits the image into patches of size 14×14:

$$N_{\text{patches}} = \frac{H}{14} \times \frac{W}{14} = 16 \times 16 = 256 \text{ tokens}$$

### Step 2: Vision Features

The frozen CLIP encoder outputs patch-level features:

$$\mathbf{Z}_v \in \mathbb{R}^{256 \times 1024}$$

Each of the 256 patches is represented as a 1024-dimensional vector (CLIP ViT-L hidden size).

### Step 3: MLP Projector

A two-layer MLP with GELU activation maps vision features into the LLM's embedding space:

$$\mathbf{H}_v = \text{GELU}(\mathbf{Z}_v W_1) W_2$$

where $W_1 \in \mathbb{R}^{1024 \times 4096}$ and $W_2 \in \mathbb{R}^{4096 \times 4096}$.

Output: $\mathbf{H}_v \in \mathbb{R}^{256 \times 4096}$ — image tokens now live in the same 4096-dim space as LLaMA text embeddings.

### Step 4: Text Token Embeddings

Text tokens are embedded via the LLM's token embedding layer:

$$\mathbf{H}_t \in \mathbb{R}^{T \times 4096}$$

### Step 5: Concatenation

Image and text tokens are concatenated along the sequence dimension:

$$\mathbf{H} = [\mathbf{H}_v; \mathbf{H}_t] \in \mathbb{R}^{(256 + T) \times 4096}$$

### Step 6: LLM Processing

The combined sequence is fed into LLaMA-7B's decoder layers with modified causal attention (see next section). The LLM processes all $256 + T$ tokens, treating image tokens as additional context — exactly like extra-long text tokens with no special architecture changes inside the LLM itself.

## Causal Attention with Image Tokens

Standard causal (autoregressive) attention masks each token from seeing future tokens. LLaVA modifies this mask to handle image tokens, which have no inherent ordering and should be fully visible to all subsequent tokens.

**Attention mask** $M \in \mathbb{R}^{(N_{\text{img}}+T) \times (N_{\text{img}}+T)}$:

$$M_{ij} = \begin{cases} 0 & \text{if } j \leq N_{\text{img}} \text{ (image tokens visible to all)} \\ 0 & \text{if } i > N_{\text{img}} \text{ and } j \leq i \text{ (causal for text)} \\ -\infty & \text{otherwise} \end{cases}$$

where $N_{\text{img}} = 256$ is the number of image tokens and $T$ is the number of text tokens.

**Interpretation:**

| Token type | Can attend to | Cannot attend to |
|------------|--------------|------------------|
| Image token $i \leq N_{\text{img}}$ | All image tokens (bidirectional) | Future text tokens |
| Text token $i > N_{\text{img}}$ | All image tokens + previous text tokens | Future text tokens |

This is implemented by setting $M_{ij} = 0$ (allow) or $M_{ij} = -\infty$ (block) before the softmax:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right) V$$

**Why bidirectional image attention?** Image patches have no temporal ordering — patch 47 is not "before" patch 12. Allowing full image-to-image attention lets each patch aggregate context from the entire image, similar to BERT-style encoding. Text tokens then attend to this fully-contextualized image representation.

In [ ]:
# Build Mini-LLaVA from scratch

class VisionProjector(nn.Module):
    """Projects image features to LLM dimension (the trainable bridge)."""
    def __init__(self, vision_dim=512, llm_dim=256):
        super().__init__()
        self.projector = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )

    def forward(self, image_features):
        return self.projector(image_features)


class MiniVisionEncoder(nn.Module):
    """Small ViT as stand-in for CLIP vision encoder."""
    def __init__(self, img_size=32, patch_size=4, dim=128, n_layers=2):
        super().__init__()
        n_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, dim, patch_size, patch_size)
        self.pos = nn.Parameter(torch.randn(1, n_patches, dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=4, dim_feedforward=dim*4, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2) + self.pos
        return self.norm(self.enc(x))  # [B, N_patches, dim]


class MiniLLM(nn.Module):
    """Small causal language model."""
    def __init__(self, vocab_size=500, dim=256, n_layers=3, max_len=128):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_len, dim)
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=4, dim_feedforward=dim*4, batch_first=True)
        self.decoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Linear(dim, vocab_size)
        self.dim = dim

    def forward(self, token_embeds):
        """Takes pre-embedded tokens (image + text already concatenated)."""
        B, T, D = token_embeds.shape
        pos = self.pos_emb(torch.arange(T, device=token_embeds.device)).unsqueeze(0)
        x = token_embeds + pos
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        x = self.decoder(x, mask=mask)
        return self.head(x)


class MiniLLaVA(nn.Module):
    """Complete LLaVA-style model."""
    def __init__(self, vocab_size=500, vision_dim=128, llm_dim=256):
        super().__init__()
        self.vision_encoder = MiniVisionEncoder(dim=vision_dim)
        self.projector = VisionProjector(vision_dim, llm_dim)
        self.llm = MiniLLM(vocab_size=vocab_size, dim=llm_dim)

    def forward(self, images, text_ids):
        # Get image features and project
        img_features = self.vision_encoder(images)       # [B, N_patches, vision_dim]
        img_tokens = self.projector(img_features)         # [B, N_patches, llm_dim]

        # Get text embeddings
        txt_tokens = self.llm.tok_emb(text_ids)           # [B, T, llm_dim]

        # Concatenate: [image_tokens, text_tokens]
        combined = torch.cat([img_tokens, txt_tokens], dim=1)  # [B, N+T, llm_dim]

        # Run through LLM
        logits = self.llm(combined)                       # [B, N+T, vocab_size]
        return logits


model = MiniLLaVA(vocab_size=500, vision_dim=128, llm_dim=256)
count_parameters(model)

imgs = torch.randn(2, 3, 32, 32)
txt = torch.randint(0, 500, (2, 10))
out = model(imgs, txt)
print(f"\nInput: images {imgs.shape}, text {txt.shape}")
print(f"Output logits: {out.shape}")
print(f"  First {64} tokens = image, last {10} = text predictions")

## KV Cache for Efficient Inference

During autoregressive text generation, the LLM produces one token at a time. Naively recomputing self-attention over **all** previous tokens at every step is wasteful — the keys and values for past tokens never change.

**KV Cache** stores the computed Key ($K$) and Value ($V$) tensors for all previously processed tokens. At each new generation step, only the new token's $Q$, $K$, $V$ are computed and appended to the cache:

$$\text{Attention}(Q_{\text{new}}, [K_{\text{cache}}; K_{\text{new}}], [V_{\text{cache}}; V_{\text{new}}])$$

**Memory per layer:**

$$\text{KV memory} = 2 \times T \times d \times \text{sizeof(dtype)}$$

where $T$ is sequence length, $d$ is hidden dimension, and the factor of 2 accounts for both $K$ and $V$.

**For LLaMA-7B** (32 layers, $d=4096$, fp16):

$$\text{KV cache for 2048 tokens} = 2 \times 2048 \times 4096 \times 32 \times 2 \text{ bytes} \approx 1 \text{ GB}$$

**Image token overhead:** With 256 image tokens always present in the cache:

$$\text{Image KV} = \frac{256}{2048} \times 1 \text{ GB} \approx 125 \text{ MB per request}$$

This is a fixed cost per request — the image KV cache is computed once during the prefill phase and reused for every subsequent autoregressive decoding step. For batched serving, this motivates **PagedAttention** (used in vLLM) to manage KV cache memory like virtual memory pages, avoiding fragmentation.

## Two-Stage Training Details

LLaVA's training is deliberately split into two stages to avoid catastrophic forgetting in the pretrained LLM while still learning rich vision-language alignment.

### Stage 1: Feature Alignment (Pretraining the Projector)

- **What's trained:** Only the MLP projector; CLIP ViT-L/14 and LLaMA-7B are **frozen**
- **Data:** 595K image-caption pairs from CC3M (Conceptual Captions 3M)
- **Loss:** Standard autoregressive language modeling, computed **only on text tokens** (image tokens are masked out of the loss):

$$\mathcal{L} = -\sum_{t \in \text{text}} \log P(x_t \mid x_{<t}, \mathbf{H}_v)$$

- **Hyperparameters:** Learning rate = 2e-3, 1 epoch
- **Goal:** Teach the projector to map CLIP visual features into the LLM's embedding space so that image "tokens" are semantically meaningful to the language model

### Stage 2: Visual Instruction Tuning

- **What's trained:** Projector + LLM (full fine-tune or LoRA adapters)
- **Data:** 150K instruction-following pairs (multi-turn conversations about images, generated with GPT-4)
- **Loss:** Same autoregressive loss, but computed **only on assistant responses** — user prompts and image tokens are masked:

$$\mathcal{L} = -\sum_{t \in \text{response}} \log P(x_t \mid x_{<t}, \mathbf{H}_v, \mathbf{H}_{\text{prompt}})$$

- **Hyperparameters:** Learning rate = 2e-5, 1 epoch
- **LoRA config:** $r=128$, $\alpha=256$ applied to all linear layers in the LLM
- **Goal:** Teach the model to follow visual instructions — answering questions, describing scenes, reasoning about image content

**Why two stages?** Stage 1 is cheap (25M params) and establishes the vision-language bridge. Stage 2 is expensive but only needed once the bridge exists. This mirrors how humans first learn to *see* (Stage 1) then learn to *talk about what they see* (Stage 2).

In [ ]:
# Visualize the two-stage training process

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('LLaVA Two-Stage Training', fontsize=18, fontweight='bold')

# Stage 1
ax = axes[0]
ax.set_xlim(0, 8); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('Stage 1: Pretrain Projector\n(595K image-text pairs, 1 epoch)', 
             fontsize=12, fontweight='bold', color='#F39C12')

draw_architecture_block(ax, 4, 7, 5, 0.7, 'Vision Encoder (FROZEN)', '#95A5A6')
draw_architecture_block(ax, 4, 5.5, 5, 0.9, 'Projection MLP (TRAINABLE)', '#F39C12')
draw_architecture_block(ax, 4, 3.8, 5, 0.7, 'LLM (FROZEN)', '#95A5A6')
draw_architecture_block(ax, 4, 2, 5, 0.7, 'Generate caption', '#2ECC71')

for y1, y2 in [(6.6, 6.0), (5.0, 4.2), (3.4, 2.5)]:
    draw_arrow(ax, (4, y1), (4, y2))

ax.text(4, 0.8, 'Goal: Align image features\nwith LLM embedding space',
        ha='center', fontsize=10, style='italic')

# Stage 2
ax = axes[1]
ax.set_xlim(0, 8); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('Stage 2: Instruction Tuning\n(150K instruction data, 1 epoch)',
             fontsize=12, fontweight='bold', color='#2ECC71')

draw_architecture_block(ax, 4, 7, 5, 0.7, 'Vision Encoder (FROZEN)', '#95A5A6')
draw_architecture_block(ax, 4, 5.5, 5, 0.9, 'Projection MLP (TRAINABLE)', '#F39C12')
draw_architecture_block(ax, 4, 3.8, 5, 0.7, 'LLM + LoRA (TRAINABLE)', '#2ECC71')
draw_architecture_block(ax, 4, 2, 5, 0.7, 'Follow instructions', '#2ECC71')

for y1, y2 in [(6.6, 6.0), (5.0, 4.2), (3.4, 2.5)]:
    draw_arrow(ax, (4, y1), (4, y2))

ax.text(4, 0.8, 'Goal: Learn to follow\nvisual instructions (Q&A, describe, etc.)',
        ha='center', fontsize=10, style='italic')

plt.tight_layout()
plt.savefig('../assets/llava_training_stages.png', dpi=150, bbox_inches='tight')
plt.show()

## Parameter Count Breakdown

| Component | Parameters | Trainable (Stage 1) | Trainable (Stage 2) |
|-----------|-----------|--------------------|--------------------|
| CLIP ViT-L/14 | 304M | Frozen | Frozen |
| MLP Projector | 2 × 1024 × 4096 + 4096² = 25M | ✓ 25M | ✓ 25M |
| LLaMA-7B | 6.7B | Frozen | LoRA: 20M (0.3%) |
| **Total trainable** | | **25M** | **45M** |

**Projector parameter derivation:** The two-layer MLP has $W_1 \in \mathbb{R}^{1024 \times 4096}$ (4.2M params) plus bias, $W_2 \in \mathbb{R}^{4096 \times 4096}$ (16.8M params) plus bias — totaling ~25M trainable parameters. This is intentionally small: the projector is a lightweight adapter, not a full cross-modal transformer.

**LoRA efficiency:** With rank $r=128$ and scaling $\alpha=256$, LoRA adds low-rank matrices $BA$ where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$ to each linear layer. For LLaMA-7B's 32 layers × 7 projection matrices, this yields ~20M additional parameters — just 0.3% of the base model — while recovering most of the performance of full fine-tuning.

## LLaVA Variants Comparison

| Model | Vision | LLM | Resolution | Key Innovation |
|-------|--------|-----|-----------|----------------|
| LLaVA | CLIP ViT-L | Vicuna-7B | 224² | MLP projector |
| LLaVA-1.5 | CLIP ViT-L/14@336 | Vicuna-13B | 336² | Higher res, more data |
| LLaVA-NeXT | SigLIP | Qwen-72B | 672² | Dynamic high-res, AnyRes |
| LLaVA-OneVision | SigLIP-SO400M | Qwen2-72B | varies | Single model for image/video/multi-image |

Each generation increases **vision resolution**, **LLM capacity**, and **training data diversity**. LLaVA-NeXT's AnyRes strategy splits high-resolution images into multiple tiles at native aspect ratio, avoiding the distortion from naive resizing. LLaVA-OneVision unifies image, video, and multi-image inputs in a single model by treating video frames and multiple images as extended token sequences.

## Modern Multimodal LLM Comparison

| Model | Vision Encoder | LLM | Projection | Training |
|-------|---------------|-----|------------|----------|
| **LLaVA** | CLIP ViT-L | LLaMA-7B | 2-layer MLP | 2 stages |
| **LLaVA-1.5** | CLIP ViT-L/14 | Vicuna-13B | 2-layer MLP | 2 stages |
| **BLIP-2** | ViT-G | FlanT5/OPT | Q-Former | 3 stages |
| **InstructBLIP** | ViT-G | Vicuna | Q-Former | Instruction tuned |
| **GPT-4V** | Unknown | GPT-4 | Unknown | End-to-end |
| **Gemini** | Built-in | Built-in | Native | End-to-end |

**For low compute: LLaVA + QLoRA is the most accessible path.**

---
**Next:** `02_multimodal_beyond_vision.ipynb` - Audio, video, and beyond